In [ ]:
!nvidia-smi

Tue Sep  8 16:23:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd "/content/drive/MyDrive/mini project train and test/swin_project"

/content/drive/MyDrive/mini project train and test/swin_project


In [ ]:
!pwd
!ls


/content/drive/MyDrive/mini project train and test/swin_project
check_dataset.py  dataset.py  model.py	__pycache__  train.py


In [ ]:
!python check_dataset.py


TRAIN DATASET
Number of classes: 13

bacterial_leaf_blight          550
bacterial_leaf_streak          429
bacterial_panicle_blight       382
black_stem_borer               430
blast                          1998
brown_spot                     1068
downy_mildew                   737
hispa                          1828
leaf_roller                    930
normal                         2044
tungro                         1658
white_stem_borer               1082
yellow_stem_borer              651

Total images: 13787

TEST DATASET
Number of classes: 13

bacterial_leaf_blight          98
bacterial_leaf_streak          76
bacterial_panicle_blight       68
black_stem_borer               76
blast                          353
brown_spot                     189
downy_mildew                   131
hispa                          323
leaf_roller                    165
normal                         361
tungro                         293
white_stem_borer               191
yellow_stem_borer          

In [ ]:
!python model.py

SWIN TRANSFORMER-TINY MODEL TEST

Device: cuda
NVIDIA GPU (CUDA) is being used.

Loading Swin Transformer-Tiny...
Downloading: "https://download.pytorch.org/models/swin_t-704ceda3.pth" to /root/.cache/torch/hub/checkpoints/swin_t-704ceda3.pth
100% 108M/108M [00:00<00:00, 206MB/s]
Model loaded successfully.

Running test forward pass...

Input shape : torch.Size([32, 3, 256, 256])
Output shape: torch.Size([32, 13])

✅ Output shape is correct!
✅ Swin Transformer-Tiny is working correctly!

Total parameters     : 27,529,351
Trainable parameters : 27,529,351

MODEL TEST COMPLETED


In [ ]:
!grep -nE "MAX_EPOCHS|BATCH_SIZE|LEARNING_RATE|EARLY_STOPPING|GRADIENT_ACCUMULATION" train.py

28:    BATCH_SIZE,
39:MAX_EPOCHS = 75
410:        BATCH_SIZE
510:        MAX_EPOCHS + 1
515:            f"Epoch {epoch}/{MAX_EPOCHS}"
901:            BATCH_SIZE,
910:            MAX_EPOCHS,


In [ ]:
!grep -nE "BATCH_SIZE|MAX_EPOCHS|LEARNING_RATE|EARLY_STOPPING_PATIENCE|GRADIENT_ACCUMULATION_STEPS" train.py dataset.py

train.py:28:    BATCH_SIZE,
train.py:39:MAX_EPOCHS = 75
train.py:410:        BATCH_SIZE
train.py:510:        MAX_EPOCHS + 1
train.py:515:            f"Epoch {epoch}/{MAX_EPOCHS}"
train.py:901:            BATCH_SIZE,
train.py:910:            MAX_EPOCHS,
dataset.py:18:BATCH_SIZE = 32
dataset.py:296:        batch_size=BATCH_SIZE,
dataset.py:303:        batch_size=BATCH_SIZE,
dataset.py:310:        batch_size=BATCH_SIZE,


In [ ]:
!grep -nE "LEARNING_RATE|LR|PATIENCE|EARLY_STOPPING" train.py

12:from torch.optim.lr_scheduler import ReduceLROnPlateau
41:INITIAL_LR = 1e-3
43:PATIENCE = 15
47:LR_FACTOR = 0.5
49:LR_PATIENCE = 5
51:MIN_LR = 1e-6
473:    # Same LR as reference CNN
478:        lr=INITIAL_LR
485:    scheduler = ReduceLROnPlateau(
488:        factor=LR_FACTOR,
489:        patience=LR_PATIENCE,
490:        min_lr=MIN_LR
631:                PATIENCE
640:            >= PATIENCE
913:            INITIAL_LR,
922:            PATIENCE,
925:            LR_FACTOR,
928:            LR_PATIENCE,
931:            MIN_LR,


In [ ]:
!python train.py

Classes:
0: bacterial_leaf_blight
1: bacterial_leaf_streak
2: bacterial_panicle_blight
3: black_stem_borer
4: blast
5: brown_spot
6: downy_mildew
7: hispa
8: leaf_roller
9: normal
10: tungro
11: white_stem_borer
12: yellow_stem_borer

Number of classes: 13
SWIN TRANSFORMER-TINY - EXPERIMENT A

Device: cuda
Using NVIDIA GPU through CUDA.

Loading dataset...

Loading training dataset...
Total original training images: 13787

Dataset split:
-----------------------------
Training images   : 11035
Validation images : 2752
Test images       : 2439

Dataset information
-----------------------------
Training batches   : 345
Validation batches : 86
Test batches       : 77
Number of classes  : 13
Batch size         : 32

Calculating class weights...
Traceback (most recent call last):
  File "/content/drive/MyDrive/mini project train and test/swin_project/train.py", line 1025, in <module>
    main()
    ~~~~^^
  File "/content/drive/MyDrive/mini project train and test/swin_project/train.py", line

In [ ]:
# Fix class-weight calculation so it does NOT load every image

path = "train.py"

with open(path, "r") as f:
    code = f.read()

start = code.index("def calculate_class_weights")
end = code.index("\ndef ", start + 5)

new_function = '''def calculate_class_weights(train_loader):
    """Calculate class weights directly from labels without loading images."""
    from collections import Counter
    import torch

    labels = [label for _, label in train_loader.dataset.samples]
    counts = Counter(labels)

    total = len(labels)
    num_classes = len(counts)

    weights = []
    for class_idx in range(num_classes):
        count = counts[class_idx]
        weight = total / (num_classes * count)
        weights.append(weight)

    return torch.tensor(weights, dtype=torch.float32)
'''

code = code[:start] + new_function + code[end:]

with open(path, "w") as f:
    f.write(code)

print("✅ Class-weight calculation fixed!")

✅ Class-weight calculation fixed!


In [ ]:
!grep -n -A20 "def calculate_class_weights" train.py

137:def calculate_class_weights(train_loader):
138-    """Calculate class weights directly from labels without loading images."""
139-    from collections import Counter
140-    import torch
141-
142-    labels = [label for _, label in train_loader.dataset.samples]
143-    counts = Counter(labels)
144-
145-    total = len(labels)
146-    num_classes = len(counts)
147-
148-    weights = []
149-    for class_idx in range(num_classes):
150-        count = counts[class_idx]
151-        weight = total / (num_classes * count)
152-        weights.append(weight)
153-
154-    return torch.tensor(weights, dtype=torch.float32)
155-
156-def train_one_epoch(
157-    model,


In [ ]:
!python -u train.py


Classes:
0: bacterial_leaf_blight
1: bacterial_leaf_streak
2: bacterial_panicle_blight
3: black_stem_borer
4: blast
5: brown_spot
6: downy_mildew
7: hispa
8: leaf_roller
9: normal
10: tungro
11: white_stem_borer
12: yellow_stem_borer

Number of classes: 13
SWIN TRANSFORMER-TINY - EXPERIMENT A

Device: cuda
Using NVIDIA GPU through CUDA.

Loading dataset...

Loading training dataset...
Total original training images: 13787

Dataset split:
-----------------------------
Training images   : 11035
Validation images : 2752
Test images       : 2439

Dataset information
-----------------------------
Training batches   : 345
Validation batches : 86
Test batches       : 77
Number of classes  : 13
Batch size         : 32

Calculating class weights...
Traceback (most recent call last):
  File "/content/drive/MyDrive/mini project train and test/swin_project/train.py", line 991, in <module>
    main()
    ~~~~^^
  File "/content/drive/MyDrive/mini project train and test/swin_project/train.py", line 

In [ ]:
path = "train.py"

with open(path, "r") as f:
    code = f.read()

old = '    return torch.tensor(weights, dtype=torch.float32)'

new = '''    class_weights = torch.tensor(weights, dtype=torch.float32)
    class_counts = [counts[i] for i in range(num_classes)]

    return class_weights, class_counts'''

if old in code:
    code = code.replace(old, new, 1)
    with open(path, "w") as f:
        f.write(code)
    print("✅ Fixed class_weights + class_counts return!")
else:
    print("❌ Expected return line not found. Do not run training yet.")

✅ Fixed class_weights + class_counts return!


In [ ]:
!grep -n -A25 "def calculate_class_weights" train.py

137:def calculate_class_weights(train_loader):
138-    """Calculate class weights directly from labels without loading images."""
139-    from collections import Counter
140-    import torch
141-
142-    labels = [label for _, label in train_loader.dataset.samples]
143-    counts = Counter(labels)
144-
145-    total = len(labels)
146-    num_classes = len(counts)
147-
148-    weights = []
149-    for class_idx in range(num_classes):
150-        count = counts[class_idx]
151-        weight = total / (num_classes * count)
152-        weights.append(weight)
153-
154-    class_weights = torch.tensor(weights, dtype=torch.float32)
155-    class_counts = [counts[i] for i in range(num_classes)]
156-
157-    return class_weights, class_counts
158-
159-def train_one_epoch(
160-    model,
161-    loader,
162-    criterion,


In [ ]:
!python -u train.py

Classes:
0: bacterial_leaf_blight
1: bacterial_leaf_streak
2: bacterial_panicle_blight
3: black_stem_borer
4: blast
5: brown_spot
6: downy_mildew
7: hispa
8: leaf_roller
9: normal
10: tungro
11: white_stem_borer
12: yellow_stem_borer

Number of classes: 13
SWIN TRANSFORMER-TINY - EXPERIMENT A

Device: cuda
Using NVIDIA GPU through CUDA.

Loading dataset...

Loading training dataset...
Total original training images: 13787

Dataset split:
-----------------------------
Training images   : 11035
Validation images : 2752
Test images       : 2439

Dataset information
-----------------------------
Training batches   : 345
Validation batches : 86
Test batches       : 77
Number of classes  : 13
Batch size         : 32

Calculating class weights...

Class weights
-----------------------------
bacterial_leaf_blight          count=  440 weight=1.9292
bacterial_leaf_streak          count=  344 weight=2.4676
bacterial_panicle_blight       count=  306 weight=2.7740
black_stem_borer               cou

In [ ]:
!find /content/train -type f | wc -l

9307


In [ ]:
!rsync -ah --info=progress2 \
"/content/drive/MyDrive/mini project train and test/train/" \
"/content/train/"

          1.12G 100%  945.22kB/s    0:19:15 (xfr#13788, to-chk=0/13802)


In [ ]:
!rsync -ah --info=progress2 \
"/content/drive/MyDrive/mini project train and test/test/" \
"/content/test/"

        197.92M 100%  340.97kB/s    0:09:26 (xfr#2440, to-chk=0/2454)


In [ ]:
!find /content/train -type f | wc -l
!find /content/test -type f | wc -l

13788
2440


In [ ]:
!sed -i 's|TRAIN_DIR = "../train"|TRAIN_DIR = "/content/train"|' dataset.py
!sed -i 's|TEST_DIR = "../test"|TEST_DIR = "/content/test"|' dataset.py

In [ ]:
!grep -nE "TRAIN_DIR|TEST_DIR|BATCH_SIZE" dataset.py

14:TRAIN_DIR = "/content/train"
15:TEST_DIR = "/content/test"
18:BATCH_SIZE = 32
43:    for folder in os.listdir(TRAIN_DIR)
44:    if os.path.isdir(os.path.join(TRAIN_DIR, folder))
98:            TRAIN_DIR,
169:            TEST_DIR,
296:        batch_size=BATCH_SIZE,
303:        batch_size=BATCH_SIZE,
310:        batch_size=BATCH_SIZE,


In [ ]:
!python check_dataset.py


TRAIN DATASET
Number of classes: 13

bacterial_leaf_blight          550
bacterial_leaf_streak          429
bacterial_panicle_blight       382
black_stem_borer               430
blast                          1998
brown_spot                     1068
downy_mildew                   737
hispa                          1828
leaf_roller                    930
normal                         2044
tungro                         1658
white_stem_borer               1082
yellow_stem_borer              651

Total images: 13787

TEST DATASET
Number of classes: 13

bacterial_leaf_blight          98
bacterial_leaf_streak          76
bacterial_panicle_blight       68
black_stem_borer               76
blast                          353
brown_spot                     189
downy_mildew                   131
hispa                          323
leaf_roller                    165
normal                         361
tungro                         293
white_stem_borer               191
yellow_stem_borer          

In [ ]:
import time
import torch
from dataset import create_dataloaders
from model import SwinTinyClassifier

device = torch.device("cuda")

train_loader, val_loader, test_loader = create_dataloaders()

print("Getting first batch...")
start = time.time()

images, labels = next(iter(train_loader))

print("First batch loaded!")
print("Images:", images.shape)
print("Labels:", labels.shape)
print("Time:", round(time.time() - start, 2), "seconds")

model = SwinTinyClassifier(num_classes=13).to(device)
model.train()

images = images.to(device)
labels = labels.to(device)

print("Running forward pass...")
start = time.time()

outputs = model(images)

print("Forward pass completed!")
print("Output:", outputs.shape)
print("Time:", round(time.time() - start, 2), "seconds")

Classes:
0: bacterial_leaf_blight
1: bacterial_leaf_streak
2: bacterial_panicle_blight
3: black_stem_borer
4: blast
5: brown_spot
6: downy_mildew
7: hispa
8: leaf_roller
9: normal
10: tungro
11: white_stem_borer
12: yellow_stem_borer

Number of classes: 13

Loading training dataset...
Total original training images: 13787

Dataset split:
-----------------------------
Training images   : 11035
Validation images : 2752
Test images       : 2439
Getting first batch...
First batch loaded!
Images: torch.Size([32, 3, 256, 256])
Labels: torch.Size([32])
Time: 0.28 seconds
Running forward pass...
Forward pass completed!
Output: torch.Size([32, 13])
Time: 0.59 seconds


In [ ]:
!sed -n '160,300p' train.py

    model,
    loader,
    criterion,
    optimizer,
    device
):

    model.train()

    running_loss = 0.0

    correct = 0

    total = 0

    for images, labels in loader:

        images = images.to(device)

        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += (
            loss.item() *
            images.size(0)
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

    epoch_loss = (
        running_loss / total
    )

    epoch_accuracy = (
        correct / total
    )

    return (
        epoch_loss,
        epoch_accuracy
    )


# ============================================================
# VALIDATE / TEST
# ==============

In [ ]:
import time
import torch
from dataset import create_dataloaders
from model import SwinTinyClassifier

device = torch.device("cuda")

train_loader, _, _ = create_dataloaders()

images, labels = next(iter(train_loader))
images = images.to(device)
labels = labels.to(device)

model = SwinTinyClassifier(num_classes=13).to(device)
model.train()

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print("Running one training batch...")

start = time.time()

optimizer.zero_grad()
outputs = model(images)
loss = criterion(outputs, labels)

print("Forward + loss:", round(time.time() - start, 2), "seconds")

start = time.time()

loss.backward()

print("Backward:", round(time.time() - start, 2), "seconds")

start = time.time()

optimizer.step()

print("Optimizer step:", round(time.time() - start, 2), "seconds")

print("Total loss:", loss.item())
print("✅ One complete training batch finished.")


Loading training dataset...
Total original training images: 13787

Dataset split:
-----------------------------
Training images   : 11035
Validation images : 2752
Test images       : 2439
Running one training batch...
Forward + loss: 0.27 seconds
Backward: 0.43 seconds
Optimizer step: 0.03 seconds
Total loss: 2.443101406097412
✅ One complete training batch finished.


In [ ]:
# Create a corrected full copy of train.py
# Keeps your existing code and only adds visible batch progress.

from pathlib import Path

path = Path("train.py")
code = path.read_text()

old = """    for images, labels in loader:

        images = images.to(device)

        labels = labels.to(device)

        optimizer.zero_grad()
"""

new = """    total_batches = len(loader)

    for batch_idx, (images, labels) in enumerate(loader, start=1):

        if batch_idx == 1 or batch_idx % 10 == 0 or batch_idx == total_batches:
            print(
                f"Processing batch {batch_idx}/{total_batches}...",
                flush=True
            )

        images = images.to(device)

        labels = labels.to(device)

        optimizer.zero_grad()
"""

if old not in code:
    print("❌ Could not find the expected training-loop section.")
    print("Your train.py was NOT changed.")
else:
    code = code.replace(old, new, 1)
    path.write_text(code)

    print("✅ Full train.py updated successfully.")
    print("✅ Existing training/evaluation code was preserved.")
    print("✅ Batch progress will now be displayed.")

✅ Full train.py updated successfully.
✅ Existing training/evaluation code was preserved.
✅ Batch progress will now be displayed.


In [ ]:
!sed -n '155,230p' train.py

    class_counts = [counts[i] for i in range(num_classes)]

    return class_weights, class_counts

def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device
):

    model.train()

    running_loss = 0.0

    correct = 0

    total = 0

    total_batches = len(loader)

    for batch_idx, (images, labels) in enumerate(loader, start=1):

        if batch_idx == 1 or batch_idx % 10 == 0 or batch_idx == total_batches:
            print(
                f"Processing batch {batch_idx}/{total_batches}...",
                flush=True
            )

        images = images.to(device)

        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += (
            loss.item() *
            images.size(0)
        )

        predictions = torch.argmax(
            outputs,
            dim

In [ ]:
!sed -i 's|BATCH_SIZE = 32|BATCH_SIZE = 8|' dataset.py

In [ ]:
!grep -n "BATCH_SIZE" dataset.py

18:BATCH_SIZE = 8
296:        batch_size=BATCH_SIZE,
303:        batch_size=BATCH_SIZE,
310:        batch_size=BATCH_SIZE,


In [ ]:
!python -u train.py

Classes:
0: bacterial_leaf_blight
1: bacterial_leaf_streak
2: bacterial_panicle_blight
3: black_stem_borer
4: blast
5: brown_spot
6: downy_mildew
7: hispa
8: leaf_roller
9: normal
10: tungro
11: white_stem_borer
12: yellow_stem_borer

Number of classes: 13
SWIN TRANSFORMER-TINY - EXPERIMENT A

Device: cuda
Using NVIDIA GPU through CUDA.

Loading dataset...

Loading training dataset...
Total original training images: 13787

Dataset split:
-----------------------------
Training images   : 11035
Validation images : 2752
Test images       : 2439

Dataset information
-----------------------------
Training batches   : 1380
Validation batches : 344
Test batches       : 305
Number of classes  : 13
Batch size         : 8

Calculating class weights...

Class weights
-----------------------------
bacterial_leaf_blight          count=  440 weight=1.9292
bacterial_leaf_streak          count=  344 weight=2.4676
bacterial_panicle_blight       count=  306 weight=2.7740
black_stem_borer               c

In [ ]:
!python swin_project/train.py

python3: can't open file '/content/drive/MyDrive/mini project train and test/swin_project/swin_project/train.py': [Errno 2] No such file or directory


In [ ]:
!pwd
!ls

/content/drive/MyDrive/mini project train and test/swin_project
check_dataset.py  dataset.py  __pycache__  train.py
checkpoints	  model.py    results


In [ ]:
!nano train.py

/bin/bash: line 1: nano: command not found


In [ ]:
%cd "/content/drive/MyDrive/mini project train and test/swin_project"

/content/drive/MyDrive/mini project train and test/swin_project


In [ ]:
%cd "/content/drive/MyDrive/mini project train and test/swin_project"
!grep -n "INIT_LR\|FREEZE_EPOCHS\|ACCUMULATION_STEPS" train.py

/content/drive/MyDrive/mini project train and test/swin_project
29:INIT_LR = 1e-4
38:FREEZE_EPOCHS = 3
42:ACCUMULATION_STEPS = 4
325:    print("Gradient accumulation:", ACCUMULATION_STEPS)
328:        BATCH_SIZE * ACCUMULATION_STEPS
403:        lr=INIT_LR
442:        if epoch == FREEZE_EPOCHS + 1:
454:                lr=INIT_LR
475:            ACCUMULATION_STEPS
761:            BATCH_SIZE * ACCUMULATION_STEPS,
763:        "learning_rate": INIT_LR,


In [ ]:
!ls -lh "/content/drive/MyDrive/mini project train and test/swin_project/checkpoints/best_swin_A.pth"

-rw------- 1 root root 316M Sep  8 19:38 '/content/drive/MyDrive/mini project train and test/swin_project/checkpoints/best_swin_A.pth'


In [ ]:
%cd "/content/drive/MyDrive/mini project train and test/swin_project"

!pwd
!ls -lh checkpoints/best_swin_A.pth
!ls -lh resume_train.py

/content/drive/MyDrive/mini project train and test/swin_project
/content/drive/MyDrive/mini project train and test/swin_project
-rw------- 1 root root 316M Sep  8 19:39 checkpoints/best_swin_A.pth
-rw------- 1 root root 12K Sep 11 06:46 resume_train.py


In [ ]:
!ls -lh resume_train.py

-rw------- 1 root root 12K Sep 11 06:46 resume_train.py


In [ ]:
!python -m py_compile resume_train.py

In [ ]:
!python -c "from dataset import TRAIN_DIR, TEST_DIR; import os; print(TRAIN_DIR); print(os.path.exists(TRAIN_DIR)); print(TEST_DIR); print(os.path.exists(TEST_DIR))"

Classes:
0: bacterial_leaf_blight
1: bacterial_leaf_streak
2: bacterial_panicle_blight
3: black_stem_borer
4: blast
5: brown_spot
6: downy_mildew
7: hispa
8: leaf_roller
9: normal
10: tungro
11: white_stem_borer
12: yellow_stem_borer

Number of classes: 13
/content/drive/MyDrive/mini project train and test/train
True
/content/drive/MyDrive/mini project train and test/test
True


In [ ]:
!python -m py_compile resume_train.py

In [ ]:
!nvidia-smi

Fri Sep 11 07:29:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   70C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!ls "/content/drive/MyDrive/mini project train and test/swin_project/checkpoints/best_swin_A.pth"

'/content/drive/MyDrive/mini project train and test/swin_project/checkpoints/best_swin_A.pth'


In [ ]:
!python resume_train.py

Classes:
0: bacterial_leaf_blight
1: bacterial_leaf_streak
2: bacterial_panicle_blight
3: black_stem_borer
4: blast
5: brown_spot
6: downy_mildew
7: hispa
8: leaf_roller
9: normal
10: tungro
11: white_stem_borer
12: yellow_stem_borer

Number of classes: 13
RESUMING SWIN TRANSFORMER-TINY — CONFIGURATION A

Device: cuda
NVIDIA GPU (CUDA) is being used.

Loading dataset...

Loading training dataset...
Total original training images: 13787

Dataset split:
-----------------------------
Training images   : 11035
Validation images : 2752
Test images       : 2439

Dataset loaded successfully.

Calculating class weights...

Class counts:
 0  bacterial_leaf_blight           440
 1  bacterial_leaf_streak           344
 2  bacterial_panicle_blight        306
 3  black_stem_borer                344
 4  blast                           1599
 5  brown_spot                      855
 6  downy_mildew                    590
 7  hispa                           1463
 8  leaf_roller                     744
 

In [ ]:
!nvidia-smi

Fri Sep 11 07:44:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!rm -rf /content/train /content/test

!cp -r "/content/drive/MyDrive/mini project train and test/train" /content/train
!cp -r "/content/drive/MyDrive/mini project train and test/test" /content/test

print("TRAIN IMAGES:")
!find /content/train -type f | wc -l

print("TEST IMAGES:")
!find /content/test -type f | wc -l

TRAIN IMAGES:
13788
TEST IMAGES:
2440


In [ ]:
!find /content/train -type f | wc -l
!du -sh /content/train

13788
1.1G	/content/train


In [ ]:
%cd "/content/drive/MyDrive/mini project train and test/swin_project"

!python dataset.py

/content/drive/MyDrive/mini project train and test/swin_project
Classes:
0: bacterial_leaf_blight
1: bacterial_leaf_streak
2: bacterial_panicle_blight
3: black_stem_borer
4: blast
5: brown_spot
6: downy_mildew
7: hispa
8: leaf_roller
9: normal
10: tungro
11: white_stem_borer
12: yellow_stem_borer

Number of classes: 13

TESTING DATA PIPELINE

Loading training dataset...
Total original training images: 13787

Dataset split:
-----------------------------
Training images   : 11035
Validation images : 2752
Test images       : 2439

Batch information:
-----------------------------
Image batch shape : torch.Size([8, 3, 256, 256])
Label batch shape : torch.Size([8])

Expected image shape:
(batch_size, 3, 256, 256)

First few labels:
tensor([ 4,  7,  5, 10,  0, 11,  6, 11])

✅ Dataset pipeline working!


In [ ]:
!ls -lh "/content/drive/MyDrive/mini project train and test/swin_project/checkpoints/"

total 631M
-rw------- 1 root root 316M Sep  8 19:39 best_swin_A.pth
-rw------- 1 root root 316M Sep  8 17:39 swin_A_best.pth


In [ ]:
%cd "/content/drive/MyDrive/mini project train and test/swin_project"

!python dataset.py

/content/drive/MyDrive/mini project train and test/swin_project
Classes:
0: bacterial_leaf_blight
1: bacterial_leaf_streak
2: bacterial_panicle_blight
3: black_stem_borer
4: blast
5: brown_spot
6: downy_mildew
7: hispa
8: leaf_roller
9: normal
10: tungro
11: white_stem_borer
12: yellow_stem_borer

Number of classes: 13

TESTING DATA PIPELINE

Loading training dataset...
Total original training images: 13787

Dataset split:
-----------------------------
Training images   : 11035
Validation images : 2752
Test images       : 2439

Batch information:
-----------------------------
Image batch shape : torch.Size([8, 3, 256, 256])
Label batch shape : torch.Size([8])

Expected image shape:
(batch_size, 3, 256, 256)

First few labels:
tensor([ 4,  7,  5, 10,  0, 11,  6, 11])

✅ Dataset pipeline working!


In [ ]:
%cd "/content/drive/MyDrive/mini project train and test/swin_project"

!grep -n "TRAIN_DIR\|TEST_DIR" dataset.py

/content/drive/MyDrive/mini project train and test/swin_project
13:TRAIN_DIR = "/content/train"
14:TEST_DIR = "/content/test"
41:    for folder in os.listdir(TRAIN_DIR)
42:    if os.path.isdir(os.path.join(TRAIN_DIR, folder))
96:            TRAIN_DIR,
167:            TEST_DIR,


In [ ]:
print("LOCAL TRAIN:")
!find /content/train -type f | wc -l

print("LOCAL TEST:")
!find /content/test -type f | wc -l

print("\nDRIVE TRAIN:")
!find "/content/drive/MyDrive/mini project train and test/train" -type f | wc -l

print("DRIVE TEST:")
!find "/content/drive/MyDrive/mini project train and test/test" -type f | wc -l

LOCAL TRAIN:
13788
LOCAL TEST:
2440

DRIVE TRAIN:
13788
DRIVE TEST:
2440


In [ ]:
%cd "/content/drive/MyDrive/mini project train and test/swin_project"

!python dataset.py

/content/drive/MyDrive/mini project train and test/swin_project
Classes:
0: bacterial_leaf_blight
1: bacterial_leaf_streak
2: bacterial_panicle_blight
3: black_stem_borer
4: blast
5: brown_spot
6: downy_mildew
7: hispa
8: leaf_roller
9: normal
10: tungro
11: white_stem_borer
12: yellow_stem_borer

Number of classes: 13

TESTING DATA PIPELINE

Loading training dataset...
Total original training images: 13787

Dataset split:
-----------------------------
Training images   : 11035
Validation images : 2752
Test images       : 2439

Batch information:
-----------------------------
Image batch shape : torch.Size([8, 3, 256, 256])
Label batch shape : torch.Size([8])

Expected image shape:
(batch_size, 3, 256, 256)

First few labels:
tensor([ 4,  7,  5, 10,  0, 11,  6, 11])

✅ Dataset pipeline working!


In [ ]:
!python resume_train.py

Classes:
0: bacterial_leaf_blight
1: bacterial_leaf_streak
2: bacterial_panicle_blight
3: black_stem_borer
4: blast
5: brown_spot
6: downy_mildew
7: hispa
8: leaf_roller
9: normal
10: tungro
11: white_stem_borer
12: yellow_stem_borer

Number of classes: 13
RESUMING SWIN TRANSFORMER-TINY — CONFIGURATION A

Device: cuda
NVIDIA GPU (CUDA) is being used.

Loading dataset...

Loading training dataset...
Total original training images: 13787

Dataset split:
-----------------------------
Training images   : 11035
Validation images : 2752
Test images       : 2439

Dataset loaded successfully.

Calculating class weights...

Class counts:
 0  bacterial_leaf_blight           440
 1  bacterial_leaf_streak           344
 2  bacterial_panicle_blight        306
 3  black_stem_borer                344
 4  blast                           1599
 5  brown_spot                      855
 6  downy_mildew                    590
 7  hispa                           1463
 8  leaf_roller                     744
 

In [ ]:
%cd "/content/drive/MyDrive/mini project train and test/swin_project"

!grep -n "test_loader\|classification_report\|confusion_matrix\|post_training\|evaluate" train.py

/content/drive/MyDrive/mini project train and test/swin_project
14:    classification_report,
15:    confusion_matrix
197:def evaluate(
319:    train_loader, validation_loader, test_loader = (
482:        validation_results = evaluate(
670:    test_results = evaluate(
672:        test_loader,
707:    report = classification_report(
719:        "classification_report.csv"
730:    cm = confusion_matrix(
743:        "confusion_matrix.csv"
783:        "post_training_metrics.json"


In [ ]:
%cd "/content/drive/MyDrive/mini project train and test/swin_project"

!sed -n '640,800p' train.py

/content/drive/MyDrive/mini project train and test/swin_project
    print("=" * 70)

    checkpoint = torch.load(
        BEST_MODEL_PATH,
        map_location=device
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    print(
        "Best epoch:",
        checkpoint["epoch"]
    )

    print(
        "Best validation loss:",
        f"{checkpoint['val_loss']:.4f}"
    )

    # ========================================================
    # TEST
    # ========================================================

    print()
    print("=" * 70)
    print("FINAL TEST EVALUATION")
    print("=" * 70)

    test_results = evaluate(
        model,
        test_loader,
        criterion,
        device
    )

    print()
    print(
        f"Test Accuracy: "
        f"{test_results['accuracy']:.4f}"
    )

    print(
        f"Test Balanced Accuracy: "
        f"{test_results['balanced_accuracy']:.4f}"
    )

    print(
        f"Test Macro F1: "
        f"{test_res

In [ ]:
%cd "/content/drive/MyDrive/mini project train and test/swin_project"

!grep -n "BEST_MODEL_PATH" train.py

/content/drive/MyDrive/mini project train and test/swin_project
49:BEST_MODEL_PATH = os.path.join(
575:                BEST_MODEL_PATH
643:        BEST_MODEL_PATH,
854:        f"✓ {BEST_MODEL_PATH}"


In [ ]:
%cd "/content/drive/MyDrive/mini project train and test/swin_project"

!sed -n '45,55p' train.py

/content/drive/MyDrive/mini project train and test/swin_project

CHECKPOINT_DIR = "checkpoints"
RESULTS_DIR = "results"

BEST_MODEL_PATH = os.path.join(
    CHECKPOINT_DIR,
    "best_swin_A.pth"
)

FINAL_MODEL_PATH = os.path.join(
    CHECKPOINT_DIR,


In [ ]:
%cd "/content/drive/MyDrive/mini project train and test/swin_project"

!python evaluate_test.py

/content/drive/MyDrive/mini project train and test/swin_project
Classes:
0: bacterial_leaf_blight
1: bacterial_leaf_streak
2: bacterial_panicle_blight
3: black_stem_borer
4: blast
5: brown_spot
6: downy_mildew
7: hispa
8: leaf_roller
9: normal
10: tungro
11: white_stem_borer
12: yellow_stem_borer

Number of classes: 13
FINAL TEST EVALUATION — SWIN TRANSFORMER-TINY (Configuration A)

Device: cuda

Loading dataset...

Loading training dataset...
Total original training images: 13787

Dataset split:
-----------------------------
Training images   : 11035
Validation images : 2752
Test images       : 2439
Test images: 2439

Loading best checkpoint...
Best checkpoint epoch: 24
Best checkpoint val_loss: 0.0460064597882561

Running evaluation on TEST set...

TEST RESULTS
Loss              : 0.0516
Accuracy          : 0.9836
Balanced Accuracy : 0.9854
Macro F1          : 0.9840
MCC               : 0.9818
Cohen's Kappa     : 0.9818

Per-class report saved to: results/test_classification_report.c

In [ ]:
import pandas as pd

metadata = pd.read_csv("/content/metadata.csv")

print("Shape:", metadata.shape)
print("Columns:", metadata.columns.tolist())
print()
print(metadata.head())

Shape: (16225, 4)
Columns: ['image_id', 'label', 'variety', 'age']

       image_id                  label         variety  age
0  PDD00001.jpg  bacterial_leaf_blight              45   65
1  PDD00002.jpg  bacterial_leaf_blight              45   60
2  PDD00003.jpg  bacterial_leaf_blight              45   55
3  PDD00004.jpg  bacterial_leaf_blight  KarnatakaPonni   50
4  PDD00005.jpg  bacterial_leaf_blight              45   72


In [ ]:
import os

all_images = []

for root, dirs, files in os.walk("/content/train"):
    for f in files:
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            all_images.append(f)

for root, dirs, files in os.walk("/content/test"):
    for f in files:
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            all_images.append(f)

metadata_ids = set(metadata["image_id"].astype(str))
image_ids = set(all_images)

print("Total image files:", len(image_ids))
print("Total metadata records:", len(metadata_ids))

print("\nImages WITHOUT metadata:")
missing_images = sorted(image_ids - metadata_ids)
print(missing_images)

print("\nMetadata records WITHOUT image:")
missing_metadata = sorted(metadata_ids - image_ids)
print(missing_metadata)

Total image files: 0
Total metadata records: 16225

Images WITHOUT metadata:
[]

Metadata records WITHOUT image:
['PDD00001.jpg', 'PDD00002.jpg', 'PDD00003.jpg', 'PDD00004.jpg', 'PDD00005.jpg', 'PDD00006.jpg', 'PDD00007.jpg', 'PDD00008.jpg', 'PDD00009.jpg', 'PDD00010.jpg', 'PDD00011.jpg', 'PDD00012.jpg', 'PDD00013.jpg', 'PDD00014.jpg', 'PDD00015.jpg', 'PDD00016.jpg', 'PDD00017.jpg', 'PDD00018.jpg', 'PDD00019.jpg', 'PDD00020.jpg', 'PDD00021.jpg', 'PDD00022.jpg', 'PDD00023.jpg', 'PDD00024.jpg', 'PDD00025.jpg', 'PDD00026.jpg', 'PDD00027.jpg', 'PDD00028.jpg', 'PDD00029.jpg', 'PDD00030.jpg', 'PDD00031.jpg', 'PDD00032.jpg', 'PDD00033.jpg', 'PDD00034.jpg', 'PDD00035.jpg', 'PDD00036.jpg', 'PDD00037.jpg', 'PDD00038.jpg', 'PDD00039.jpg', 'PDD00040.jpg', 'PDD00041.jpg', 'PDD00042.jpg', 'PDD00043.jpg', 'PDD00044.jpg', 'PDD00045.jpg', 'PDD00046.jpg', 'PDD00047.jpg', 'PDD00048.jpg', 'PDD00049.jpg', 'PDD00050.jpg', 'PDD00051.jpg', 'PDD00052.jpg', 'PDD00053.jpg', 'PDD00054.jpg', 'PDD00055.jpg', 'PDD00

In [ ]:
import os

base = "/content/drive/MyDrive/mini project train and test"

print("Folders/files inside project folder:\n")

for item in os.listdir(base):
    print(item)

Folders/files inside project folder:

.DS_Store
swin_project
train
test


In [ ]:
import os

BASE = "/content/drive/MyDrive/mini project train and test"

all_images = []

for root, dirs, files in os.walk(os.path.join(BASE, "train")):
    for f in files:
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            all_images.append(f)

for root, dirs, files in os.walk(os.path.join(BASE, "test")):
    for f in files:
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            all_images.append(f)

metadata_ids = set(metadata["image_id"].astype(str))
image_ids = set(all_images)

print("Total image files:", len(image_ids))
print("Total metadata records:", len(metadata_ids))

print("\nImages WITHOUT metadata:")
print(sorted(image_ids - metadata_ids))

print("\nMetadata records WITHOUT image:")
print(sorted(metadata_ids - image_ids))

Total image files: 16226
Total metadata records: 16225

Images WITHOUT metadata:
['PDD15461 - Copy.jpg']

Metadata records WITHOUT image:
[]


In [ ]:
import os

BASE = "/content/drive/MyDrive/mini project train and test"

target = "PDD15461 - Copy.jpg"

print("Locations of the extra image:")

for split in ["train", "test"]:
    for root, dirs, files in os.walk(os.path.join(BASE, split)):
        if target in files:
            print(os.path.join(root, target))

print("\nLocations of PDD15461.jpg:")

original = "PDD15461.jpg"

for split in ["train", "test"]:
    for root, dirs, files in os.walk(os.path.join(BASE, split)):
        if original in files:
            print(os.path.join(root, original))

Locations of the extra image:
/content/drive/MyDrive/mini project train and test/train/yellow_stem_borer/PDD15461 - Copy.jpg

Locations of PDD15461.jpg:
/content/drive/MyDrive/mini project train and test/train/yellow_stem_borer/PDD15461.jpg


In [ ]:
import os
import pandas as pd

BASE = "/content/drive/MyDrive/mini project train and test"

metadata = pd.read_csv("/content/metadata.csv")

# Create image -> folder label mapping
image_folder_labels = {}

for split in ["train", "test"]:
    split_dir = os.path.join(BASE, split)

    for class_name in os.listdir(split_dir):
        class_dir = os.path.join(split_dir, class_name)

        if not os.path.isdir(class_dir):
            continue

        for f in os.listdir(class_dir):
            if f.lower().endswith((".jpg", ".jpeg", ".png")):
                image_folder_labels[f] = class_name

# Compare metadata labels with folder labels
mismatches = []

for _, row in metadata.iterrows():
    image_id = str(row["image_id"])
    metadata_label = str(row["label"])

    folder_label = image_folder_labels.get(image_id)

    if folder_label is None:
        continue

    if metadata_label != folder_label:
        mismatches.append(
            (image_id, metadata_label, folder_label)
        )

print("Total metadata records:", len(metadata))
print("Images found:", len(image_folder_labels))
print("Label mismatches:", len(mismatches))

if mismatches:
    print("\nFirst mismatches:")
    for item in mismatches[:20]:
        print(item)
else:
    print("\n✓ All metadata labels match their image folders.")

Total metadata records: 16225
Images found: 16226
Label mismatches: 0

✓ All metadata labels match their image folders.


In [ ]:
import os
import py_compile

PROJECT = "/content/drive/MyDrive/mini project train and test/swin_project"

files = [
    "dataset_B.py",
    "model_B.py",
    "train_B.py"
]

for f in files:
    path = os.path.join(PROJECT, f)
    print(f"\nChecking: {f}")

    if not os.path.exists(path):
        print("❌ FILE NOT FOUND")
    else:
        try:
            py_compile.compile(path, doraise=True)
            print("✅ Syntax OK")
        except Exception as e:
            print("❌ ERROR:")
            print(e)


Checking: dataset_B.py
✅ Syntax OK

Checking: model_B.py
✅ Syntax OK

Checking: train_B.py
✅ Syntax OK


In [ ]:
import os

PROJECT = "/content/drive/MyDrive/mini project train and test/swin_project"

for filename in ["dataset_B.py", "model_B.py", "train_B.py"]:
    path = os.path.join(PROJECT, filename)

    print("\n" + "=" * 60)
    print(filename)
    print("=" * 60)

    print("File size:", os.path.getsize(path), "bytes")

    with open(path, "r", encoding="utf-8") as f:
        content = f.read()

    print("Number of characters:", len(content))
    print("\nFirst 1500 characters:\n")
    print(content[:1500])


dataset_B.py
File size: 0 bytes
Number of characters: 0

First 1500 characters:



model_B.py
File size: 0 bytes
Number of characters: 0

First 1500 characters:



train_B.py
File size: 0 bytes
Number of characters: 0

First 1500 characters:




In [ ]:
print("Missing values:")
print(metadata.isnull().sum())

print("\nUnique varieties:")
print(metadata["variety"].value_counts(dropna=False))

print("\nAge range:")
print("Minimum:", metadata["age"].min())
print("Maximum:", metadata["age"].max())

Missing values:
image_id    0
label       0
variety     0
age         0
dtype: int64

Unique varieties:
variety
45                10978
KarnatakaPonni     1404
Ponni               975
AtchayaPonni        706
Zonal               649
AndraPonni          615
Onthanel            585
IR20                235
Surya                42
RR                   36
Name: count, dtype: int64

Age range:
Minimum: 45
Maximum: 80


In [ ]:
import sys
import importlib

PROJECT = "/content/drive/MyDrive/mini project train and test/swin_project"

if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

import dataset_B
importlib.reload(dataset_B)

print("Reloaded dataset_B successfully.")
print("Number of classes:", dataset_B.NUM_CLASSES)
print("Number of varieties:", dataset_B.NUM_VARIETIES)
print("Age range:", dataset_B.AGE_MIN, "to", dataset_B.AGE_MAX)
print("Classes:", dataset_B.CLASS_NAMES)
print("Varieties:", dataset_B.VARIETIES)

Reloaded dataset_B successfully.
Number of classes: 13
Number of varieties: 10
Age range: 45.0 to 80.0
Classes: ['bacterial_leaf_blight', 'bacterial_leaf_streak', 'bacterial_panicle_blight', 'black_stem_borer', 'blast', 'brown_spot', 'downy_mildew', 'hispa', 'leaf_roller', 'normal', 'tungro', 'white_stem_borer', 'yellow_stem_borer']
Varieties: ['45', 'AndraPonni', 'AtchayaPonni', 'IR20', 'KarnatakaPonni', 'Onthanel', 'Ponni', 'RR', 'Surya', 'Zonal']


In [ ]:
import sys

PROJECT = "/content/drive/MyDrive/mini project train and test/swin_project"

if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

import dataset_B

print("Import successful.")
print("Number of classes:", dataset_B.NUM_CLASSES)
print("Number of varieties:", dataset_B.NUM_VARIETIES)
print("Age range:", dataset_B.AGE_MIN, "to", dataset_B.AGE_MAX)
print("Classes:", dataset_B.CLASS_NAMES)
print("Varieties:", dataset_B.VARIETIES)

Import successful.
Number of classes: 13
Number of varieties: 10
Age range: 45.0 to 80.0
Classes: ['bacterial_leaf_blight', 'bacterial_leaf_streak', 'bacterial_panicle_blight', 'black_stem_borer', 'blast', 'brown_spot', 'downy_mildew', 'hispa', 'leaf_roller', 'normal', 'tungro', 'white_stem_borer', 'yellow_stem_borer']
Varieties: ['45', 'AndraPonni', 'AtchayaPonni', 'IR20', 'KarnatakaPonni', 'Onthanel', 'Ponni', 'RR', 'Surya', 'Zonal']


In [ ]:
print("Rows where variety is numeric:")

numeric_variety = metadata[
    metadata["variety"].astype(str).str.fullmatch(r"\d+")
]

print(numeric_variety.to_string(index=False))

print("\nCount:", len(numeric_variety))

Rows where variety is numeric:
    image_id                    label variety  age
PDD00001.jpg    bacterial_leaf_blight      45   65
PDD00002.jpg    bacterial_leaf_blight      45   60
PDD00003.jpg    bacterial_leaf_blight      45   55
PDD00005.jpg    bacterial_leaf_blight      45   72
PDD00006.jpg    bacterial_leaf_blight      45   68
PDD00008.jpg    bacterial_leaf_blight      45   55
PDD00009.jpg    bacterial_leaf_blight      45   72
PDD00010.jpg    bacterial_leaf_blight      45   55
PDD00011.jpg    bacterial_leaf_blight      45   72
PDD00012.jpg    bacterial_leaf_blight      45   72
PDD00013.jpg    bacterial_leaf_blight      45   50
PDD00014.jpg    bacterial_leaf_blight      45   72
PDD00015.jpg    bacterial_leaf_blight      45   72
PDD00017.jpg    bacterial_leaf_blight      45   55
PDD00018.jpg    bacterial_leaf_blight      45   55
PDD00019.jpg    bacterial_leaf_blight      45   72
PDD00020.jpg    bacterial_leaf_blight      45   55
PDD00021.jpg    bacterial_leaf_blight      45   72


In [ ]:
print("\nVariety + Age combinations for the numeric variety:")
print(
    metadata[
        metadata["variety"].astype(str).str.fullmatch(r"\d+")
    ][["image_id", "label", "variety", "age"]].head(20)
)


Variety + Age combinations for the numeric variety:
        image_id                  label variety  age
0   PDD00001.jpg  bacterial_leaf_blight      45   65
1   PDD00002.jpg  bacterial_leaf_blight      45   60
2   PDD00003.jpg  bacterial_leaf_blight      45   55
4   PDD00005.jpg  bacterial_leaf_blight      45   72
5   PDD00006.jpg  bacterial_leaf_blight      45   68
7   PDD00008.jpg  bacterial_leaf_blight      45   55
8   PDD00009.jpg  bacterial_leaf_blight      45   72
9   PDD00010.jpg  bacterial_leaf_blight      45   55
10  PDD00011.jpg  bacterial_leaf_blight      45   72
11  PDD00012.jpg  bacterial_leaf_blight      45   72
12  PDD00013.jpg  bacterial_leaf_blight      45   50
13  PDD00014.jpg  bacterial_leaf_blight      45   72
14  PDD00015.jpg  bacterial_leaf_blight      45   72
16  PDD00017.jpg  bacterial_leaf_blight      45   55
17  PDD00018.jpg  bacterial_leaf_blight      45   55
18  PDD00019.jpg  bacterial_leaf_blight      45   72
19  PDD00020.jpg  bacterial_leaf_blight      4

In [ ]:
import sys
import importlib

PROJECT = "/content/drive/MyDrive/mini project train and test/swin_project"

if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

import dataset_B
importlib.reload(dataset_B)

print("dataset_B reloaded.")

dataset_B reloaded.


In [ ]:
train_loader, val_loader, test_loader = dataset_B.create_dataloaders()

batch = next(iter(train_loader))

print("\n========== BATCH TEST ==========")
print("Image shape:", batch["image"].shape)
print("Variety shape:", batch["variety"].shape)
print("Age shape:", batch["age"].shape)
print("Label shape:", batch["label"].shape)

print("\nVariety indices:", batch["variety"])
print("Normalized ages:", batch["age"])
print("Labels:", batch["label"])
print("Image IDs:", batch["image_id"])


Images found before metadata filtering: 13787

Images skipped because metadata is unavailable:
   PDD15461 - Copy.jpg
Images with metadata: 13786

Creating stratified train/validation split...
Training samples: 11034
Validation samples: 2752

Test images before metadata filtering: 2439
Test images with metadata: 2439

========== BATCH TEST ==========
Image shape: torch.Size([8, 3, 256, 256])
Variety shape: torch.Size([8])
Age shape: torch.Size([8])
Label shape: torch.Size([8])

Variety indices: tensor([0, 0, 1, 0, 0, 1, 0, 4])
Normalized ages: tensor([0.7143, 0.6286, 0.5714, 0.7143, 0.4286, 0.5714, 0.0000, 0.8571])
Labels: tensor([12,  4, 12,  4,  9, 12,  7,  4])
Image IDs: ['PDD15583.jpg', 'PDD02423.jpg', 'PDD15680.jpg', 'PDD02998.jpg', 'PDD10297.jpg', 'PDD15608.jpg', 'PDD07727.jpg', 'PDD03706.jpg']


In [ ]:
import sys
import importlib
import torch

PROJECT = "/content/drive/MyDrive/mini project train and test/swin_project"

if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

import dataset_B
import model_B

importlib.reload(dataset_B)
importlib.reload(model_B)

print("dataset_B loaded.")
print("model_B loaded.")

device = model_B.get_device()

print("Device:", device)

train_loader, val_loader, test_loader = (
    dataset_B.create_dataloaders()
)

batch = next(iter(train_loader))

model = model_B.SwinMetadataClassifier(
    num_classes=dataset_B.NUM_CLASSES,
    num_varieties=dataset_B.NUM_VARIETIES
)

model = model.to(device)

images = batch["image"].to(device)
varieties = batch["variety"].to(device)
ages = batch["age"].to(device)

with torch.no_grad():

    outputs = model(
        images,
        varieties,
        ages
    )

print("\n========== MODEL TEST ==========")

print("Input image shape:", images.shape)
print("Variety shape:", varieties.shape)
print("Age shape:", ages.shape)
print("Output shape:", outputs.shape)

print(
    "Total parameters:",
    sum(p.numel() for p in model.parameters())
)

print("\nFirst output:")
print(outputs[0])

dataset_B loaded.
model_B loaded.
Device: cuda

Images found before metadata filtering: 13787

Images skipped because metadata is unavailable:
   PDD15461 - Copy.jpg
Images with metadata: 13786

Creating stratified train/validation split...
Training samples: 11034
Validation samples: 2752

Test images before metadata filtering: 2439
Test images with metadata: 2439
Downloading: "https://download.pytorch.org/models/swin_t-704ceda3.pth" to /root/.cache/torch/hub/checkpoints/swin_t-704ceda3.pth


100%|██████████| 108M/108M [00:00<00:00, 192MB/s] 



========== MODEL TEST ==========
Input image shape: torch.Size([8, 3, 256, 256])
Variety shape: torch.Size([8])
Age shape: torch.Size([8])
Output shape: torch.Size([8, 13])
Total parameters: 27999175

First output:
tensor([ 1.9999e-01, -8.2107e-01,  3.6100e-01,  5.6517e-01, -3.1547e-01,
        -1.4618e-01, -2.9492e-01,  1.2589e-01, -3.1131e-01,  7.2838e-01,
         1.4759e-01,  8.1379e-04, -3.0834e-01], device='cuda:0')


In [ ]:
import py_compile

path = "/content/drive/MyDrive/mini project train and test/swin_project/train_B.py"

py_compile.compile(
    path,
    doraise=True
)

print("✅ train_B.py syntax OK")

✅ train_B.py syntax OK


In [ ]:
import os

PROJECT = "/content/drive/MyDrive/mini project train and test/swin_project"

print("Checking Config B files:\n")

for f in [
    "dataset_B.py",
    "model_B.py",
    "train_B.py"
]:
    path = os.path.join(PROJECT, f)
    print(
        f"{f}:",
        "✅ exists" if os.path.exists(path) else "❌ missing"
    )

print("\nChecking Config A best checkpoint:")

a_checkpoint = os.path.join(
    PROJECT,
    "checkpoints",
    "best_swin_A.pth"
)

print(
    "best_swin_A.pth:",
    "✅ exists" if os.path.exists(a_checkpoint) else "❌ missing"
)

print("\nChecking Config B checkpoint:")

b_checkpoint = os.path.join(
    PROJECT,
    "checkpoints",
    "best_swin_B.pth"
)

print(
    "best_swin_B.pth:",
    "already exists" if os.path.exists(b_checkpoint)
    else "not created yet — correct"
)

Checking Config B files:

dataset_B.py: ✅ exists
model_B.py: ✅ exists
train_B.py: ✅ exists

Checking Config A best checkpoint:
best_swin_A.pth: ✅ exists

Checking Config B checkpoint:
best_swin_B.pth: not created yet — correct


In [ ]:
import shutil
import os

source = "/content/metadata.csv"
destination = "/content/drive/MyDrive/mini project train and test/swin_project/metadata.csv"

shutil.copy(source, destination)

print("✅ metadata.csv copied to:")
print(destination)

print("File exists:", os.path.exists(destination))

✅ metadata.csv copied to:
/content/drive/MyDrive/mini project train and test/swin_project/metadata.csv
File exists: True


In [ ]:
import py_compile

path = "/content/drive/MyDrive/mini project train and test/swin_project/train_B.py"

py_compile.compile(
    path,
    doraise=True
)

print("✅ train_B.py syntax OK")

✅ train_B.py syntax OK


In [ ]:
%cd "/content/drive/MyDrive/mini project train and test/swin_project"

!python -m py_compile train_B.py

print("✅ train_B.py syntax OK")

/content/drive/MyDrive/mini project train and test/swin_project
✅ train_B.py syntax OK


In [ ]:
!python train_B.py

SWIN TRANSFORMER-TINY — CONFIGURATION B
IMAGE + CROP METADATA

Device: cuda
NVIDIA GPU / CUDA is being used.

LOADING CONFIGURATION B DATA

Images found before metadata filtering: 13787

Images skipped because metadata is unavailable:
   PDD15461 - Copy.jpg
Images with metadata: 13786

Creating stratified train/validation split...
Training samples: 11034
Validation samples: 2752

Test images before metadata filtering: 2439
Test images with metadata: 2439

Dataset information:
-----------------------------
Training samples   : 11034
Validation samples : 2752
Test samples       : 2439
Number of classes  : 13
Batch size         : 8
Gradient accumulation: 4
Effective batch size: 32

Calculating class weights...

Class counts and weights:
-----------------------------
 0 | bacterial_leaf_blight          |   440 | weight = 1.9290
 1 | bacterial_leaf_streak          |   344 | weight = 2.4674
 2 | bacterial_panicle_blight       |   306 | weight = 2.7738
 3 | black_stem_borer               |   

In [ ]:
!ls /content/train | wc -l

ls: cannot access '/content/train': No such file or directory
0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import shutil

DRIVE_ROOT = "/content/drive/MyDrive/mini project train and test"
PROJECT_DIR = os.path.join(DRIVE_ROOT, "swin_project")

print("Checking Google Drive...")

print("Drive root exists:",
      os.path.exists(DRIVE_ROOT))

print("Train exists:",
      os.path.exists(os.path.join(DRIVE_ROOT, "train")))

print("Test exists:",
      os.path.exists(os.path.join(DRIVE_ROOT, "test")))

print("Project exists:",
      os.path.exists(PROJECT_DIR))

print("Metadata exists:",
      os.path.exists(os.path.join(PROJECT_DIR, "metadata.csv")))

Checking Google Drive...
Drive root exists: True
Train exists: True
Test exists: True
Project exists: True
Metadata exists: True


In [ ]:
import shutil
import os

DRIVE_ROOT = "/content/drive/MyDrive/mini project train and test"
PROJECT_DIR = os.path.join(DRIVE_ROOT, "swin_project")

# Copy train
if not os.path.exists("/content/train"):
    print("Copying train dataset...")
    shutil.copytree(
        os.path.join(DRIVE_ROOT, "train"),
        "/content/train"
    )
else:
    print("/content/train already exists")

# Copy test
if not os.path.exists("/content/test"):
    print("Copying test dataset...")
    shutil.copytree(
        os.path.join(DRIVE_ROOT, "test"),
        "/content/test"
    )
else:
    print("/content/test already exists")

# Make sure metadata is inside project folder
metadata_source = "/content/metadata.csv"
metadata_destination = os.path.join(
    PROJECT_DIR,
    "metadata.csv"
)

if not os.path.exists(metadata_destination):
    if os.path.exists(metadata_source):
        shutil.copy(
            metadata_source,
            metadata_destination
        )
        print("Metadata copied to project folder.")
    else:
        print("⚠️ /content/metadata.csv not found.")
else:
    print("Metadata already exists in project folder.")

print("\nSetup complete.")

Copying train dataset...
Copying test dataset...
Metadata already exists in project folder.

Setup complete.


In [ ]:
!find /content/train -type f | wc -l
!find /content/test -type f | wc -l

13788
2440


In [ ]:
import os

test_dir = "/content/test"

for root, dirs, files in os.walk(test_dir):
    for f in files:
        print(os.path.join(root, f))

/content/test/.DS_Store
/content/test/bacterial_panicle_blight/PDD01457.jpg
/content/test/bacterial_panicle_blight/PDD01367.jpg
/content/test/bacterial_panicle_blight/PDD01529.jpg
/content/test/bacterial_panicle_blight/PDD01583.jpg
/content/test/bacterial_panicle_blight/PDD01186.jpg
/content/test/bacterial_panicle_blight/PDD01278.jpg
/content/test/bacterial_panicle_blight/PDD01156.jpg
/content/test/bacterial_panicle_blight/PDD01371.jpg
/content/test/bacterial_panicle_blight/PDD01239.jpg
/content/test/bacterial_panicle_blight/PDD01282.jpg
/content/test/bacterial_panicle_blight/PDD01398.jpg
/content/test/bacterial_panicle_blight/PDD01485.jpg
/content/test/bacterial_panicle_blight/PDD01484.jpg
/content/test/bacterial_panicle_blight/PDD01506.jpg
/content/test/bacterial_panicle_blight/PDD01173.jpg
/content/test/bacterial_panicle_blight/PDD01362.jpg
/content/test/bacterial_panicle_blight/PDD01168.jpg
/content/test/bacterial_panicle_blight/PDD01602.jpg
/content/test/bacterial_panicle_blight/P

In [ ]:
import os

test_dir = "/content/test"

image_ext = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

non_images = []

for root, dirs, files in os.walk(test_dir):
    for f in files:
        ext = os.path.splitext(f)[1].lower()

        if ext not in image_ext:
            non_images.append(
                os.path.join(root, f)
            )

print("Non-image files:", len(non_images))

for f in non_images:
    print(f)

Non-image files: 1
/content/test/.DS_Store


In [ ]:
%cd "/content/drive/MyDrive/mini project train and test/swin_project"

/content/drive/MyDrive/mini project train and test/swin_project


In [ ]:
%cd "/content/drive/MyDrive/mini project train and test/swin_project"

!python -m py_compile train_B.py

print("✅ train_B.py syntax OK")

/content/drive/MyDrive/mini project train and test/swin_project
✅ train_B.py syntax OK


In [ ]:
import shutil
import os

source = "/content/drive/MyDrive/mini project train and test/swin_project/metadata.csv"
destination = "/content/metadata.csv"

print("Source exists:", os.path.exists(source))

shutil.copy(source, destination)

print("Metadata copied:", os.path.exists(destination))

Source exists: True
Metadata copied: True


In [ ]:
import pandas as pd

metadata = pd.read_csv("/content/metadata.csv")

print("Metadata shape:", metadata.shape)
print("Columns:", metadata.columns.tolist())
print(metadata.head())

Metadata shape: (16225, 4)
Columns: ['image_id', 'label', 'variety', 'age']
       image_id                  label         variety  age
0  PDD00001.jpg  bacterial_leaf_blight              45   65
1  PDD00002.jpg  bacterial_leaf_blight              45   60
2  PDD00003.jpg  bacterial_leaf_blight              45   55
3  PDD00004.jpg  bacterial_leaf_blight  KarnatakaPonni   50
4  PDD00005.jpg  bacterial_leaf_blight              45   72


In [ ]:
%cd "/content/drive/MyDrive/mini project train and test/swin_project"

!python train_B.py

/content/drive/MyDrive/mini project train and test/swin_project
SWIN TRANSFORMER-TINY — CONFIGURATION B
IMAGE + CROP METADATA

Device: cuda
GPU: Tesla T4
NVIDIA GPU / CUDA is being used.

LOADING CONFIGURATION B DATA

Images found before metadata filtering: 13787

Images skipped because metadata is unavailable:
   PDD15461 - Copy.jpg
Images with metadata: 13786

Creating stratified train/validation split...
Training samples: 11034
Validation samples: 2752

Test images before metadata filtering: 2439
Test images with metadata: 2439

Dataset information:
-----------------------------
Training samples   : 11034
Validation samples : 2752
Test samples       : 2439
Number of classes  : 13
Batch size         : 8
Gradient accumulation: 4
Effective batch size: 32

Calculating class weights...

Class counts and weights:
-----------------------------
 0 | bacterial_leaf_blight          |   440 | weight = 1.9290
 1 | bacterial_leaf_streak          |   344 | weight = 2.4674
 2 | bacterial_panicle_b

In [2]:
import os

DRIVE = "/content/drive/MyDrive"
targets = {"swin_project", "test", "test_images", "train"}

print("Searching Google Drive...\n")

for root, dirs, files in os.walk(DRIVE):
    for folder in dirs:
        if folder.lower() in targets:
            print("FOLDER:", os.path.join(root, folder))

    for file in files:
        if file.lower().endswith((".pth", ".pt", ".ckpt")):
            print("CHECKPOINT:", os.path.join(root, file))

    # Skip common irrelevant folders to make the search faster
    dirs[:] = [d for d in dirs if d not in {
        ".ipynb_checkpoints", "node_modules", ".cache"
    }]

Searching Google Drive...

FOLDER: /content/drive/MyDrive/test_images
FOLDER: /content/drive/MyDrive/mini project train and test/swin_project
FOLDER: /content/drive/MyDrive/mini project train and test/train
FOLDER: /content/drive/MyDrive/mini project train and test/test
FOLDER: /content/drive/MyDrive/mini project train and test/test_images
CHECKPOINT: /content/drive/MyDrive/mini project train and test/swin_project/checkpoints/swin_A_best.pth
CHECKPOINT: /content/drive/MyDrive/mini project train and test/swin_project/checkpoints/best_swin_A.pth
CHECKPOINT: /content/drive/MyDrive/mini project train and test/swin_project/checkpoints/latest_swin_A.pth
CHECKPOINT: /content/drive/MyDrive/mini project train and test/swin_project/checkpoints/best_swin_B.pth
CHECKPOINT: /content/drive/MyDrive/mini project train and test/swin_project/checkpoints/latest_swin_B.pth
CHECKPOINT: /content/drive/MyDrive/mini project train and test/swin_project/checkpoints/final_swin_B.pth


In [3]:

from pathlib import Path
import os
import pandas as pd

BASE = Path("/content/drive/MyDrive/mini project train and test")
PROJECT = BASE / "swin_project"

# 1. Show project files
print("=== SWIN PROJECT FILES ===")
for p in PROJECT.rglob("*"):
    if p.is_file() and p.suffix.lower() in {
        ".py", ".ipynb", ".csv", ".json", ".yaml", ".yml"
    }:
        print(p)

# 2. Count images in each dataset folder
print("\n=== IMAGE COUNTS ===")
image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

for folder in ["train", "test", "test_images"]:
    path = BASE / folder
    if path.exists():
        count = sum(
            1 for p in path.rglob("*")
            if p.is_file() and p.suffix.lower() in image_exts
        )
        print(f"{folder}: {count} images")

# 3. Locate metadata and prediction CSV files
print("\n=== CSV FILES ===")
for p in Path("/content/drive/MyDrive").rglob("*.csv"):
    if any(x in p.name.lower() for x in
           ["metadata", "match", "prediction", "label"]):
        print(p)
        try:
            df = pd.read_csv(p, nrows=3)
            print("Columns:", df.columns.tolist())
            print(df.head(2).to_string(index=False))
        except Exception as e:
            print("Could not read:", e)


=== SWIN PROJECT FILES ===
/content/drive/MyDrive/mini project train and test/swin_project/check_dataset.py
/content/drive/MyDrive/mini project train and test/swin_project/model.py
/content/drive/MyDrive/mini project train and test/swin_project/train.py
/content/drive/MyDrive/mini project train and test/swin_project/dataset.py
/content/drive/MyDrive/mini project train and test/swin_project/resume_train.py
/content/drive/MyDrive/mini project train and test/swin_project/evaluate_test.py
/content/drive/MyDrive/mini project train and test/swin_project/dataset_B.py
/content/drive/MyDrive/mini project train and test/swin_project/model_B.py
/content/drive/MyDrive/mini project train and test/swin_project/metadata.csv
/content/drive/MyDrive/mini project train and test/swin_project/train_B.py
/content/drive/MyDrive/mini project train and test/swin_project/results/swin_A_resume_metrics.csv
/content/drive/MyDrive/mini project train and test/swin_project/results/test_classification_report.csv
/cont

In [4]:

from pathlib import Path

PROJECT = Path(
    "/content/drive/MyDrive/mini project train and test/swin_project"
)

# Print the complete Config 2 model and dataset code
for filename in ["model_B.py", "dataset_B.py"]:
    path = PROJECT / filename
    print(f"\n{'='*25} {filename} {'='*25}")
    print(path.read_text())

# Print relevant training lines for checkpoint/model setup
path = PROJECT / "train_B.py"
print("\n========== train_B.py: model/checkpoint setup ==========")

lines = path.read_text().splitlines()
keywords = [
    "Swin", "Model(", "Dataset(", "DataLoader",
    "load_state_dict", "state_dict", "checkpoint",
    "class_to_idx", "variety", "age", "transform",
    "best_swin", "final_swin", "torch.save"
]

for i, line in enumerate(lines):
    if any(word.lower() in line.lower() for word in keywords):
        start = max(0, i - 2)
        end = min(len(lines), i + 3)
        print(f"\n--- lines {start+1}-{end} ---")
        for j in range(start, end):
            print(f"{j+1}: {lines[j]}")



========================= model_B.py =========================
import torch
import torch.nn as nn

from torchvision.models import swin_t, Swin_T_Weights


# ============================================================
# SWIN-B: IMAGE + CROP METADATA
# ============================================================

class SwinMetadataClassifier(nn.Module):

    def __init__(
        self,
        num_classes=13,
        num_varieties=10,
        variety_embedding_dim=16,
        crop_feature_dim=32
    ):

        super().__init__()

        # ----------------------------------------------------
        # SWIN TRANSFORMER-TINY
        # ----------------------------------------------------

        weights = Swin_T_Weights.IMAGENET1K_V1

        self.swin = swin_t(
            weights=weights
        )

        # Swin-Tiny feature dimension
        image_feature_dim = (
            self.swin.head.in_features
        )

        # Remove original ImageNet classification head
        self.swi

In [6]:

import pandas as pd
from pathlib import Path

BASE = Path("/content/drive/MyDrive/mini project train and test")
PROJECT = BASE / "swin_project"

files_to_check = [
    BASE / "swin_A_known_image_matches.csv",
    PROJECT / "results" / "swin_A_new_images_predictions.csv",
    PROJECT / "metadata.csv",
]

for path in files_to_check:
    print("\n" + "=" * 70)
    print("FILE:", path)

    if not path.exists():
        print("Not found")
        continue

    try:
        df = pd.read_csv(path)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        print(df.head(10).to_string(index=False))
    except Exception as e:
        print("Could not read:", e)



FILE: /content/drive/MyDrive/mini project train and test/swin_A_known_image_matches.csv
Could not read: No columns to parse from file

FILE: /content/drive/MyDrive/mini project train and test/swin_project/results/swin_A_new_images_predictions.csv
Shape: (3469, 3)
Columns: ['image', 'predicted_class', 'confidence_percent']
     image  predicted_class  confidence_percent
200001.jpg            hispa              100.00
200002.jpg           normal              100.00
200003.jpg            blast               99.66
200004.jpg            blast              100.00
200005.jpg            blast              100.00
200006.jpg       brown_spot               99.67
200007.jpg white_stem_borer              100.00
200008.jpg       brown_spot              100.00
200009.jpg            hispa               99.99
200010.jpg           normal              100.00

FILE: /content/drive/MyDrive/mini project train and test/swin_project/metadata.csv
Shape: (16225, 4)
Columns: ['image_id', 'label', 'variety', 'ag

In [7]:

from pathlib import Path
from collections import defaultdict
import hashlib
import pandas as pd

BASE = Path("/content/drive/MyDrive/mini project train and test")
NEW_DIR = BASE / "test_images"
SOURCE_DIRS = [BASE / "train", BASE / "test"]

EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

# Index the original train/test images by their file content
source_hashes = defaultdict(list)

for folder in SOURCE_DIRS:
    for path in folder.rglob("*"):
        if path.is_file() and path.suffix.lower() in EXTS:
            digest = sha256_file(path)
            source_hashes[digest].append({
                "source_file": path.name,
                "source_path": str(path),
                "true_class_from_folder": path.parent.name
            })

# Match each new image against the source index
matches = []
unmatched = []

for path in sorted(NEW_DIR.rglob("*")):
    if not path.is_file() or path.suffix.lower() not in EXTS:
        continue

    digest = sha256_file(path)
    found = source_hashes.get(digest, [])

    if found:
        for item in found:
            matches.append({
                "new_image": path.name,
                **item
            })
    else:
        unmatched.append(path.name)

match_df = pd.DataFrame(matches)
match_path = BASE / "swin_project" / "results" / "test_images_exact_matches.csv"
match_df.to_csv(match_path, index=False)

print("New images checked:", len(list(
    p for p in NEW_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in EXTS
)))
print("Images with exact content matches:", len(set(
    m["new_image"] for m in matches
)))
print("Images without exact matches:", len(unmatched))
print("\nSample matches:")
print(match_df.head(15).to_string(index=False) if not match_df.empty else "No exact matches")
print("\nSaved mapping:", match_path)


New images checked: 3469
Images with exact content matches: 0
Images without exact matches: 3469

Sample matches:
No exact matches

Saved mapping: /content/drive/MyDrive/mini project train and test/swin_project/results/test_images_exact_matches.csv


In [8]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/mini project train and test")
IMAGE_DIR = BASE / "test_images"

print("Top-level entries:")
for p in list(IMAGE_DIR.iterdir())[:25]:
    print(p.name, "— folder" if p.is_dir() else "— file")

print("\nFirst 30 image filenames:")
images = sorted(
    p.name for p in IMAGE_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in
    {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
)
print(images[:30])

Top-level entries:
202482.jpg — file
202334.jpg — file
202542.jpg — file
202396.jpg — file
202520.jpg — file
202408.jpg — file
202539.jpg — file
202379.jpg — file
202525.jpg — file
202342.jpg — file
202367.jpg — file
202468.jpg — file
202493.jpg — file
202510.jpg — file
202360.jpg — file
202442.jpg — file
202361.jpg — file
202546.jpg — file
202439.jpg — file
202354.jpg — file
202398.jpg — file
202329.jpg — file
202402.jpg — file
202364.jpg — file
202412.jpg — file

First 30 image filenames:
['200001.jpg', '200002.jpg', '200003.jpg', '200004.jpg', '200005.jpg', '200006.jpg', '200007.jpg', '200008.jpg', '200009.jpg', '200010.jpg', '200011.jpg', '200012.jpg', '200013.jpg', '200014.jpg', '200015.jpg', '200016.jpg', '200017.jpg', '200018.jpg', '200019.jpg', '200020.jpg', '200021.jpg', '200022.jpg', '200023.jpg', '200024.jpg', '200025.jpg', '200026.jpg', '200027.jpg', '200028.jpg', '200029.jpg', '200030.jpg']


In [9]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive")

for p in ROOT.rglob("*"):
    if p.is_file() and p.suffix.lower() in {
        ".csv", ".xlsx", ".xls", ".json"
    }:
        print(p)

/content/drive/MyDrive/mini project train and test/swin_A_known_image_matches.csv
/content/drive/MyDrive/mini project train and test/swin_project/metadata.csv
/content/drive/MyDrive/mini project train and test/swin_project/results/swin_A_resume_metrics.csv
/content/drive/MyDrive/mini project train and test/swin_project/results/test_classification_report.csv
/content/drive/MyDrive/mini project train and test/swin_project/results/test_confusion_matrix.csv
/content/drive/MyDrive/mini project train and test/swin_project/results/test_summary.json
/content/drive/MyDrive/mini project train and test/swin_project/results/training_metrics_B.csv
/content/drive/MyDrive/mini project train and test/swin_project/results/classification_report_B.csv
/content/drive/MyDrive/mini project train and test/swin_project/results/confusion_matrix_B.csv
/content/drive/MyDrive/mini project train and test/swin_project/results/post_training_metrics_B.json
/content/drive/MyDrive/mini project train and test/swin_proje

In [10]:
from pathlib import Path
import json
import pandas as pd

RESULTS = Path(
    "/content/drive/MyDrive/mini project train and test/"
    "swin_project/results"
)

for filename in [
    "test_summary.json",
    "post_training_metrics_B.json",
]:
    path = RESULTS / filename
    print(f"\n===== {filename} =====")

    if path.exists():
        print(json.dumps(json.loads(path.read_text()), indent=2))

for filename in [
    "test_classification_report.csv",
    "classification_report_B.csv",
]:
    path = RESULTS / filename
    print(f"\n===== {filename} =====")

    if path.exists():
        df = pd.read_csv(path)
        print(df.to_string(index=False))


===== test_summary.json =====
{
  "checkpoint_epoch": 24,
  "test_loss": 0.05155313835889052,
  "test_accuracy": 0.98359983599836,
  "test_balanced_accuracy": 0.9853514061974976,
  "test_macro_f1": 0.9840075637036085,
  "test_mcc": 0.9817706124147872,
  "test_cohen_kappa": 0.9817605073854427,
  "num_test_images": 2439
}

===== post_training_metrics_B.json =====
{
  "model": "Swin Transformer-Tiny",
  "configuration": "B - Image + Crop Metadata",
  "modalities": [
    "Leaf Image",
    "Variety",
    "Age"
  ],
  "num_classes": 13,
  "image_size": 256,
  "batch_size": 8,
  "gradient_accumulation": 4,
  "effective_batch_size": 32,
  "optimizer": "Adam",
  "learning_rate": 0.0001,
  "weight_decay": 0.0,
  "freeze_epochs": 3,
  "best_epoch": 25,
  "best_validation_loss": 0.10172334615597635,
  "test_accuracy": 0.973759737597376,
  "test_balanced_accuracy": 0.9736597276310911,
  "test_macro_f1": 0.9737823211828627,
  "test_mcc": 0.9708253836489413,
  "test_cohen_kappa": 0.9708073917353814,

In [11]:

from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/mini project train and test")
IMAGE_DIR = BASE / "test_images"
PROJECT = BASE / "swin_project"

image_ids = {
    p.name for p in IMAGE_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in
    {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
}

print("Target image count:", len(image_ids))

for path in BASE.rglob("*"):
    if not path.is_file() or path.suffix.lower() != ".csv":
        continue

    try:
        df = pd.read_csv(path)
    except Exception:
        continue

    # Find columns that might contain image filenames
    candidate_cols = [
        c for c in df.columns
        if any(word in c.lower() for word in
               ["image", "filename", "file", "path", "id"])
    ]

    for col in candidate_cols:
        values = set(df[col].dropna().astype(str).str.strip())
        overlap = image_ids & values

        if overlap:
            print("\nPossible metadata/mapping file:", path)
            print("Matching column:", col)
            print("Matched images:", len(overlap))
            print("Examples:", list(overlap)[:10])


Target image count: 3469

Possible metadata/mapping file: /content/drive/MyDrive/mini project train and test/swin_project/results/swin_A_new_images_predictions.csv
Matching column: image
Matched images: 3469
Examples: ['202933.jpg', '201448.jpg', '202499.jpg', '202514.jpg', '200558.jpg', '203135.jpg', '201379.jpg', '201902.jpg', '200820.jpg', '200216.jpg']


In [12]:
import pandas as pd
from pathlib import Path

BASE = Path("/content/drive/MyDrive/mini project train and test")
metadata = pd.read_csv(BASE / "swin_project" / "metadata.csv")

print("Original metadata examples:")
print(metadata[["image_id", "label", "variety", "age"]].head(10))

print("\nNumeric image examples:")
image_dir = BASE / "test_images"
images = sorted(
    p.name for p in image_dir.iterdir()
    if p.is_file() and p.suffix.lower() in [".jpg", ".jpeg", ".png"]
)
print(images[:20])

print("\nMetadata image ID examples:")
print(metadata["image_id"].head(20).tolist())

Original metadata examples:
       image_id                  label         variety  age
0  PDD00001.jpg  bacterial_leaf_blight              45   65
1  PDD00002.jpg  bacterial_leaf_blight              45   60
2  PDD00003.jpg  bacterial_leaf_blight              45   55
3  PDD00004.jpg  bacterial_leaf_blight  KarnatakaPonni   50
4  PDD00005.jpg  bacterial_leaf_blight              45   72
5  PDD00006.jpg  bacterial_leaf_blight              45   68
6  PDD00007.jpg  bacterial_leaf_blight           Ponni   47
7  PDD00008.jpg  bacterial_leaf_blight              45   55
8  PDD00009.jpg  bacterial_leaf_blight              45   72
9  PDD00010.jpg  bacterial_leaf_blight              45   55

Numeric image examples:
['200001.jpg', '200002.jpg', '200003.jpg', '200004.jpg', '200005.jpg', '200006.jpg', '200007.jpg', '200008.jpg', '200009.jpg', '200010.jpg', '200011.jpg', '200012.jpg', '200013.jpg', '200014.jpg', '200015.jpg', '200016.jpg', '200017.jpg', '200018.jpg', '200019.jpg', '200020.jpg']

Metad

In [13]:
!pip -q install ImageHash

from pathlib import Path
from PIL import Image
import imagehash
import pandas as pd

BASE = Path("/content/drive/MyDrive/mini project train and test")
NEW_DIR = BASE / "test_images"
SOURCE_DIRS = [BASE / "train", BASE / "test"]

extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Index original images by perceptual hash
source_hashes = []

for folder in SOURCE_DIRS:
    for p in folder.rglob("*"):
        if p.is_file() and p.suffix.lower() in extensions:
            try:
                with Image.open(p) as im:
                    h = imagehash.phash(im.convert("RGB"))
                source_hashes.append((h, p))
            except Exception:
                pass

print("Original images indexed:", len(source_hashes))

# Test a small sample first
new_images = sorted(
    p for p in NEW_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in extensions
)[:20]

for p in new_images:
    with Image.open(p) as im:
        h = imagehash.phash(im.convert("RGB"))

    nearest = sorted(
        ((h - sh, source_path) for sh, source_path in source_hashes),
        key=lambda x: x[0]
    )[:3]

    print(f"\n{p.name}")
    for distance, source_path in nearest:
        print(f"  distance={distance}: {source_path.name}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 1.7 MB/s eta 0:00:00
Original images indexed: 16226

200001.jpg
  distance=0: PDD08695.jpg
  distance=14: PDD15542.jpg
  distance=16: PDD08070.jpg

200002.jpg
  distance=2: PDD11072.jpg
  distance=6: PDD10266.jpg
  distance=8: PDD10229.jpg

200003.jpg
  distance=0: PDD03898.jpg
  distance=8: PDD03626.jpg
  distance=8: PDD02540.jpg

200004.jpg
  distance=0: PDD04338.jpg
  distance=2: PDD03769.jpg
  distance=2: PDD02269.jpg

200005.jpg
  distance=0: PDD02975.jpg
  distance=8: PDD02399.jpg
  distance=10: PDD03504.jpg

200006.jpg
  distance=0: PDD04938.jpg
  distance=4: PDD04596.jpg
  distance=6: PDD05483.jpg

200007.jpg
  distance=0: PDD14842.jpg
  distance=2: PDD14848.jpg
  distance=2: PDD15163.jpg

200008.jpg
  distance=0: PDD04784.jpg
  distance=12: PDD05114.jpg
  distance=16: PDD15944.jpg

200009.jpg
  distance=0: PDD08006.jpg
  distance=6: PDD07777.jpg
  distance=6: PDD08715.jpg

200010.jpg
  distance=2: PDD12149.jpg
  distan

In [14]:

from pathlib import Path
from collections import defaultdict
from PIL import Image
import imagehash
import pandas as pd

BASE = Path("/content/drive/MyDrive/mini project train and test")
NEW_DIR = BASE / "test_images"
SOURCE_DIRS = [BASE / "train", BASE / "test"]
PROJECT = BASE / "swin_project"
RESULTS = PROJECT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Load original image metadata
metadata = pd.read_csv(PROJECT / "metadata.csv")
metadata["image_id"] = metadata["image_id"].astype(str).str.strip()
metadata["variety"] = metadata["variety"].astype(str).str.strip()

meta_lookup = metadata.drop_duplicates("image_id").set_index("image_id")

# Hash the original Paddy Doctor images
source_hashes = []

for folder in SOURCE_DIRS:
    for p in folder.rglob("*"):
        if p.is_file() and p.suffix.lower() in EXTS:
            try:
                with Image.open(p) as im:
                    h = imagehash.phash(im.convert("RGB"))
                source_hashes.append((h, p))
            except Exception:
                pass

print("Original images indexed:", len(source_hashes))

# Match every numeric-named image
rows = []

new_images = sorted(
    p for p in NEW_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in EXTS
)

for i, new_path in enumerate(new_images, 1):
    try:
        with Image.open(new_path) as im:
            new_hash = imagehash.phash(im.convert("RGB"))

        distances = sorted(
            (new_hash - source_hash, source_path)
            for source_hash, source_path in source_hashes
        )

        best_distance = distances[0][0]
        best_candidates = [
            path for distance, path in distances
            if distance == best_distance
        ]

        # Only accept a unique exact hash match automatically.
        # Other cases are flagged for review.
        if best_distance == 0 and len(best_candidates) == 1:
            source_path = best_candidates[0]
            original_id = source_path.name
            status = "unique_exact_match"
        else:
            source_path = None
            original_id = ""
            status = (
                "ambiguous_exact_match"
                if best_distance == 0
                else "needs_review"
            )

        row = {
            "new_image": new_path.name,
            "matched_original_id": original_id,
            "best_hash_distance": best_distance,
            "number_of_best_candidates": len(best_candidates),
            "match_status": status,
            "matched_source_path": str(source_path) if source_path else "",
            "variety": "",
            "age": None,
            "original_label": "",
        }

        if source_path and original_id in meta_lookup.index:
            m = meta_lookup.loc[original_id]
            row["variety"] = str(m["variety"])
            row["age"] = m["age"]
            row["original_label"] = str(m["label"])

        rows.append(row)

    except Exception as e:
        rows.append({
            "new_image": new_path.name,
            "match_status": f"error: {e}",
        })

    if i % 250 == 0:
        print(f"Processed {i}/{len(new_images)} images")

mapping = pd.DataFrame(rows)
mapping_path = RESULTS / "test_images_metadata_mapping.csv"
mapping.to_csv(mapping_path, index=False)

print("\n=== MATCH SUMMARY ===")
print(mapping["match_status"].value_counts(dropna=False).to_string())
print("\nMetadata recovered:",
      mapping["variety"].astype(str).ne("").sum())
print("Saved mapping:", mapping_path)

print("\nSample mappings:")
print(mapping.head(20).to_string(index=False))


Original images indexed: 16226
Processed 250/3469 images
Processed 500/3469 images
Processed 750/3469 images
Processed 1000/3469 images
Processed 1250/3469 images
Processed 1500/3469 images
Processed 1750/3469 images
Processed 2000/3469 images
Processed 2250/3469 images
Processed 2500/3469 images
Processed 2750/3469 images
Processed 3000/3469 images
Processed 3250/3469 images

=== MATCH SUMMARY ===
match_status
unique_exact_match       3010
needs_review              318
ambiguous_exact_match     141

Metadata recovered: 3010
Saved mapping: /content/drive/MyDrive/mini project train and test/swin_project/results/test_images_metadata_mapping.csv

Sample mappings:
 new_image matched_original_id  best_hash_distance  number_of_best_candidates          match_status                                                                     matched_source_path        variety  age    original_label
200001.jpg        PDD08695.jpg                   0                          1    unique_exact_match      

In [16]:

# ============================================================
# SWIN TRANSFORMER-TINY — CONFIG 2 INFERENCE
# Image + Variety + Crop Age
# ============================================================

import sys
import shutil
from pathlib import Path

import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------
BASE = Path("/content/drive/MyDrive/mini project train and test")
PROJECT = BASE / "swin_project"
RESULTS = PROJECT / "results"

IMAGE_DIR = BASE / "test_images"
MAPPING_PATH = RESULTS / "test_images_metadata_mapping.csv"
CHECKPOINT_PATH = PROJECT / "checkpoints" / "best_swin_B.pth"
METADATA_PATH = PROJECT / "metadata.csv"

RESULTS.mkdir(parents=True, exist_ok=True)

assert IMAGE_DIR.exists(), f"Image folder not found: {IMAGE_DIR}"
assert MAPPING_PATH.exists(), f"Mapping not found: {MAPPING_PATH}"
assert CHECKPOINT_PATH.exists(), f"Checkpoint not found: {CHECKPOINT_PATH}"
assert METADATA_PATH.exists(), f"Metadata not found: {METADATA_PATH}"

# dataset_B.py expects metadata at /content/metadata.csv
shutil.copy2(METADATA_PATH, "/content/metadata.csv")

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from dataset_B import (
    CLASS_NAMES,
    VARIETIES,
    VARIETY_TO_INDEX,
    eval_transform,
    normalize_age,
)
from model_B import SwinMetadataClassifier

# ------------------------------------------------------------
# 2. LOAD MATCHED IMAGES AND METADATA
# ------------------------------------------------------------
mapping = pd.read_csv(MAPPING_PATH)

mapping = mapping[
    (mapping["match_status"] == "unique_exact_match")
    & mapping["matched_original_id"].notna()
    & mapping["variety"].notna()
    & mapping["age"].notna()
].copy()

mapping["variety"] = mapping["variety"].astype(str).str.strip()
mapping["age"] = pd.to_numeric(mapping["age"], errors="coerce")

mapping = mapping[
    mapping["variety"].isin(VARIETY_TO_INDEX)
    & mapping["age"].notna()
].copy()

# Verify that every image file exists
mapping = mapping[
    mapping["new_image"].apply(
        lambda name: (IMAGE_DIR / name).is_file()
    )
].copy()

mapping = mapping.reset_index(drop=True)

print("Images eligible for Config 2:", len(mapping))

if mapping.empty:
    raise ValueError("No eligible images found. Check the mapping and metadata.")

# ------------------------------------------------------------
# 3. CREATE INFERENCE DATASET
# ------------------------------------------------------------
class MatchedImageDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = IMAGE_DIR / row["new_image"]

        with Image.open(image_path) as im:
            image = eval_transform(im.convert("RGB"))

        variety_idx = VARIETY_TO_INDEX[row["variety"]]
        age_value = normalize_age(row["age"])

        return (
            image,
            torch.tensor(variety_idx, dtype=torch.long),
            torch.tensor(age_value, dtype=torch.float32),
            row["new_image"],
        )

loader = DataLoader(
    MatchedImageDataset(mapping),
    batch_size=8,
    shuffle=False,
    num_workers=0,
)

# ------------------------------------------------------------
# 4. LOAD SWIN-B MODEL AND CHECKPOINT
# ------------------------------------------------------------
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print("Device:", device)

model = SwinMetadataClassifier(
    num_classes=len(CLASS_NAMES),
    num_varieties=len(VARIETIES),
).to(device)

# IMPORTANT:
# weights_only=False is appropriate only because this is your
# own trusted checkpoint created by your training code.
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False,
)

if "model_state_dict" not in checkpoint:
    raise KeyError(
        "The checkpoint does not contain 'model_state_dict'."
    )

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Config 2 checkpoint loaded successfully!")
print("Checkpoint:", CHECKPOINT_PATH)
print("Checkpoint epoch:", checkpoint.get("epoch", "unknown"))

# ------------------------------------------------------------
# 5. RUN PREDICTIONS
# ------------------------------------------------------------
predictions = []

with torch.inference_mode():
    for batch_num, (images, varieties, ages, filenames) in enumerate(loader, 1):
        images = images.to(device)
        varieties = varieties.to(device)
        ages = ages.to(device)

        logits = model(images, varieties, ages)
        probabilities = torch.softmax(logits, dim=1)

        confidences, predicted_indices = probabilities.max(dim=1)

        for filename, confidence, pred_idx in zip(
            filenames,
            confidences.cpu().tolist(),
            predicted_indices.cpu().tolist(),
        ):
            predictions.append({
                "image": filename,
                "predicted_class": CLASS_NAMES[pred_idx],
                "confidence_percent": round(confidence * 100, 2),
            })

        if batch_num % 50 == 0 or batch_num == len(loader):
            print(
                f"Processed {min(batch_num * loader.batch_size, len(mapping))}"
                f"/{len(mapping)} images"
            )

# ------------------------------------------------------------
# 6. SAVE PREDICTIONS WITH MATCHING METADATA
# ------------------------------------------------------------
pred_df = pd.DataFrame(predictions)

pred_df = pred_df.merge(
    mapping[
        [
            "new_image",
            "matched_original_id",
            "matched_source_path",
            "variety",
            "age",
            "original_label",
        ]
    ],
    left_on="image",
    right_on="new_image",
    how="left",
).drop(columns=["new_image"])

pred_path = RESULTS / "swin_B_new_images_predictions.csv"
pred_df.to_csv(pred_path, index=False)

# ------------------------------------------------------------
# 7. SAVE PREDICTED CLASS COUNTS
# ------------------------------------------------------------
class_counts = (
    pred_df["predicted_class"]
    .value_counts()
    .reindex(CLASS_NAMES, fill_value=0)
    .rename_axis("class")
    .reset_index(name="image_count")
)

counts_path = RESULTS / "swin_B_new_images_class_counts.csv"
class_counts.to_csv(counts_path, index=False)

# ------------------------------------------------------------
# 8. SOURCE SPLIT COUNTS
# ------------------------------------------------------------
pred_df["source_split"] = pred_df["matched_source_path"].apply(
    lambda p: (
        "train" if "/train/" in str(p)
        else "test" if "/test/" in str(p)
        else "unknown"
    )
)

split_counts = (
    pred_df["source_split"]
    .value_counts()
    .rename_axis("source_split")
    .reset_index(name="image_count")
)

split_path = RESULTS / "swin_B_new_images_source_split.csv"
split_counts.to_csv(split_path, index=False)

# ------------------------------------------------------------
# 9. DISPLAY RESULTS
# ------------------------------------------------------------
print("\n=== FIRST 15 CONFIG 2 PREDICTIONS ===")
print(
    pred_df[
        ["image", "predicted_class", "confidence_percent"]
    ].head(15).to_string(index=False)
)

print("\n=== PREDICTED CLASS COUNTS ===")
print(class_counts.to_string(index=False))

print("\n=== ORIGINAL SOURCE SPLIT COUNTS ===")
print(split_counts.to_string(index=False))

print("\nTotal predictions:", len(pred_df))
print("Predictions saved:", pred_path)
print("Class counts saved:", counts_path)
print("Source split counts saved:", split_path)

# ------------------------------------------------------------
# 10. DOWNLOAD PREDICTION CSV
# ------------------------------------------------------------
from google.colab import files
files.download(str(pred_path))


Images eligible for Config 2: 3010
Device: cuda
Config 2 checkpoint loaded successfully!
Checkpoint: /content/drive/MyDrive/mini project train and test/swin_project/checkpoints/best_swin_B.pth
Checkpoint epoch: 25
Processed 400/3010 images
Processed 800/3010 images
Processed 1200/3010 images
Processed 1600/3010 images
Processed 2000/3010 images
Processed 2400/3010 images
Processed 2800/3010 images
Processed 3010/3010 images

=== FIRST 15 CONFIG 2 PREDICTIONS ===
     image  predicted_class  confidence_percent
200001.jpg            hispa               99.96
200003.jpg            blast               85.66
200004.jpg            blast               99.97
200005.jpg            blast               99.98
200006.jpg       brown_spot               99.69
200007.jpg white_stem_borer               99.98
200008.jpg       brown_spot               99.86
200009.jpg            hispa               99.95
200011.jpg white_stem_borer               93.39
200012.jpg            hispa               99.95
20001

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>